# Agents as Tools - Explicit Agent Invocation

## Purpose
Learn how to convert agents into callable tools for more explicit control over multi-agent composition. Unlike handoffs (which route automatically), this approach lets the parent agent explicitly decide when to invoke sub-agents.

## Key Concepts
- **.as_tool()**: Method that converts an agent into a function tool
- **Explicit Invocation**: Parent agent decides when to call sub-agents (vs automatic routing)
- **Tool-based Composition**: Alternative to handoffs with more control

## Installation
Install the required packages (uncomment to run):

In [ ]:
#!pip install openai
#!pip install openai-agents
#!pip install aws-bedrock-token-generator

## Authentication Setup
Configure the connection to AWS Bedrock Mantle:

In [ ]:
model_id = "openai.gpt-5.5"

In [ ]:
from openai import AsyncOpenAI
from agents import (
    set_default_openai_client,
    set_default_openai_api,
    set_tracing_disabled,
)
from aws_bedrock_token_generator import provide_token

client = AsyncOpenAI(
    api_key=provide_token(),
    base_url="https://bedrock-mantle.us-east-1.api.aws/openai/v1",
    project="default"
)

set_default_openai_client(client)
set_default_openai_api("responses")
set_tracing_disabled(True)  # OpenAI-platform tracing can't reach Mantle

## Import Libraries

In [ ]:
import asyncio

from agents import Agent, Runner

## Step 1: Create Specialist Agents

First, define your specialist agents. These are regular agents that will be converted to tools.

💡 **Note**: Unlike handoffs, these agents don't need `handoff_description` because we'll provide that when converting to tools.

In [ ]:
history_tutor = Agent(
    name="History tutor",
    model=model_id,
    instructions="Answer history questions clearly and concisely.",
)

math_tutor = Agent(
    name="Math tutor",
    model=model_id,
    instructions="Explain math step by step and include worked examples.",
)

## Step 2: Convert Agents to Tools

Use `.as_tool()` to convert each specialist into a callable tool. This gives you:
- **tool_name**: How the parent agent refers to this tool
- **tool_description**: When to use this tool (similar to handoff_description)

🔍 **Key Difference**: The parent agent treats these as tools and explicitly calls them, rather than automatic handoff routing.

In [ ]:
triage_agent = Agent(
    name="Homework triage",
    model=model_id,
    instructions="Handle all direct user communication. Route each homework question to the right specialist.",
    tools=[
        history_tutor.as_tool(
            tool_name="history_tutor",
            tool_description="Specialist for history questions.",
        ),
        math_tutor.as_tool(
            tool_name="math_tutor",
            tool_description="Specialist for math questions.",
        )
    ],
)

## Step 3: Run the System

The triage agent will treat specialists as tools and call them when appropriate.

⚡ **How it works**:
1. User question goes to triage agent
2. Triage agent decides which tool to call
3. Tool (specialist agent) processes the request
4. Result returns through triage agent to user

In [ ]:
result = await Runner.run(triage_agent, "Who was the first president of the United States?")
print(result.final_output)

## 🎉 Congratulations!

You've completed the **Agents as Tools** notebook!